[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session2/notebooks/04_augmentation.ipynb)

# session2 · 04 · 증강 (Augmentation)

- **이 노트북에서 배우는 것**: **기하 변환=라벨(박스/마스크)도 함께 움직임 / 픽셀 변환=라벨 그대로 / 나쁜 증강=해로움**.
- **입력**: 없음 — 혈액 도말 이미지 + 박스 + 마스크를 코드로 확보(이전 단계 재사용).
- **출력**: 증강 비교 그림들 + **애니메이션(GIF)** (`outputs/`) — 5단계 HTML 재사용.

> **핵심**: 증강은 *그럴듯한 변형*으로 데이터 다양성을 흉내 내는 것이지 **새 정보를 만드는 게 아니다.**
> 학습/추론/SAM2 실행 없음(3단계 산출물 재사용만). GPU 불필요. 그림 안 글자는 폰트 호환을 위해 영문.

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (00~03 과 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session2"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session2/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT, "(증강은 GPU 불필요)")

In [ ]:
# 셀 2 · 의존성 설치 (albumentations 추가). 코랩 사전설치본은 재설치 안 함.
import os, sys, subprocess

if not os.path.exists("requirements.txt"):
    raise RuntimeError("requirements.txt 를 찾지 못했습니다. 셀 1을 먼저 ▶ 실행하세요.")
print("requirements.txt 설치 중... (albumentations 포함)")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if r.returncode != 0:
    raise RuntimeError("pip install 실패. 위 로그 확인 후 이 셀(셀 2)을 다시 ▶ 실행하세요.")

print("\n[설치된 새 의존성 버전 — requirements.txt 핀에 사용]")
for mod in ["albumentations", "numpy"]:
    try:
        m = __import__(mod)
        print("  -", mod, ":", getattr(m, "__version__", "?"))
    except Exception as e:
        print("  -", mod, ": import 실패 (", e, ")")

In [ ]:
# 셀 3 · 입력 확보: 혈액 도말 이미지 + 박스(VOC) + 마스크(Otsu, 이미지와 크기 일치 보장)
import os, glob, json, subprocess
import numpy as np
from PIL import Image
import xml.etree.ElementTree as ET

demo = os.path.join("data", "_demo_inputs")
os.makedirs(demo, exist_ok=True)
AUG_IMG = os.path.join(demo, "aug_image.jpg")
AUG_MASK = os.path.join(demo, "aug_mask.png")
AUG_BOX = os.path.join(demo, "aug_boxes.json")


def otsu_threshold(gray):
    hist = np.histogram(gray, bins=256, range=(0, 256))[0].astype(float)
    total = gray.size
    sum_all = np.dot(np.arange(256), hist)
    sumB, wB, best, thr = 0.0, 0.0, -1.0, 0
    for t in range(256):
        wB += hist[t]
        if wB == 0:
            continue
        wF = total - wB
        if wF == 0:
            break
        sumB += t * hist[t]
        mB, mF = sumB / wB, (sum_all - sumB) / wF
        v = wB * wF * (mB - mF) ** 2
        if v > best:
            best, thr = v, t
    return thr


# 도말 이미지 + 매칭 XML 찾기 (1단계 data/detection 재사용, 없으면 BCCD 확보)
img_src, xml_src = None, None
det = sorted(glob.glob(os.path.join("data", "detection", "*.jpg")))
if det:
    img_src = det[0]
    stem = os.path.splitext(os.path.basename(img_src))[0]
    cand = os.path.join("data", "detection", stem + ".xml")
    xml_src = cand if os.path.exists(cand) else None
else:
    src = os.path.join("data", "_bccd_src")
    if not os.path.isdir(src):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Shenggan/BCCD_Dataset", src], check=False)
    jpgs = sorted(glob.glob(os.path.join(src, "BCCD", "JPEGImages", "*.jpg")))
    if jpgs:
        img_src = jpgs[0]
        stem = os.path.splitext(os.path.basename(img_src))[0]
        cand = os.path.join(src, "BCCD", "Annotations", stem + ".xml")
        xml_src = cand if os.path.exists(cand) else None

if img_src is None:
    raise RuntimeError("도말 이미지를 확보하지 못했습니다(네트워크). 셀 3을 다시 ▶ 실행하세요.")

img = Image.open(img_src).convert("RGB")
img.save(AUG_IMG)
W, H = img.size

# 박스 (VOC -> pascal_voc [xmin,ymin,xmax,ymax] + labels), 경계 클립
boxes, labels = [], []
if xml_src:
    for obj in ET.parse(xml_src).getroot().findall("object"):
        b = obj.find("bndbox")
        x1, y1, x2, y2 = (int(b.find(k).text) for k in ["xmin", "ymin", "xmax", "ymax"])
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
            labels.append(obj.find("name").text)
if not boxes:                       # 박스가 없으면 데모용 중앙 박스 1개
    boxes = [[int(W * 0.40), int(H * 0.40), int(W * 0.60), int(H * 0.60)]]
    labels = ["cell"]
    print("[주의] VOC 박스를 찾지 못해 데모용 중앙 박스 1개를 사용합니다(박스 출처 확인).")
# 너무 많으면 3개만(가독성)
boxes, labels = boxes[:3], labels[:3]
json.dump({"boxes": boxes, "labels": labels}, open(AUG_BOX, "w"))

# 마스크 (Otsu of smear — 전체 세포 영역; 이미지와 동일 크기 보장)
gray = np.array(img.convert("L"))
thr = otsu_threshold(gray)
mask = (gray < thr).astype(np.uint8)
Image.fromarray((mask * 255).astype(np.uint8)).save(AUG_MASK)

print("입력 확보:", os.path.basename(img_src), "| 박스", len(boxes), "개 | 마스크 shape", mask.shape, "| Otsu thr", thr)
missing = [f for f in [AUG_IMG, AUG_MASK, AUG_BOX] if not os.path.exists(f)]
if missing:
    raise RuntimeError("입력 파일 생성 실패: %s — 셀 3을 다시 ▶ 실행하세요." % missing)
print("저장 완료: data/_demo_inputs/aug_image.jpg, aug_mask.png, aug_boxes.json")

In [ ]:
# 셀 4 · 기하 증강 (라벨도 함께 변함) — image + boxes + mask 를 동시에 변형
import os, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import albumentations as A
from PIL import Image

demo = os.path.join("data", "_demo_inputs")
for f in ["aug_image.jpg", "aug_mask.png", "aug_boxes.json"]:
    if not os.path.exists(os.path.join(demo, f)):
        raise RuntimeError("입력이 없습니다. 셀 3을 먼저 ▶ 실행하세요.")
img = np.array(Image.open(os.path.join(demo, "aug_image.jpg")).convert("RGB"))
mask = (np.array(Image.open(os.path.join(demo, "aug_mask.png")).convert("L")) > 127).astype(np.uint8)
d = json.load(open(os.path.join(demo, "aug_boxes.json")))
boxes, labels = d["boxes"], d["labels"]


def show(ax, image, bxs, m, title):
    disp = image.copy()
    if m.shape == disp.shape[:2]:                       # 크기 일치할 때만 마스크 오버레이(방어)
        sel = m > 0
        disp[sel] = (0.5 * disp[sel] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
    ax.imshow(disp)
    for bx in bxs:
        x1, y1, x2, y2 = [int(v) for v in bx]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=1.5))
    ax.set_title(title)
    ax.axis("off")


# 결정적 기하 변환: 좌우 플립 + 25도 회전 (라벨도 함께 변형됨)
transform = A.Compose(
    [A.HorizontalFlip(p=1.0), A.Affine(rotate=(25, 25), p=1.0)],
    bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"]),
)
t = transform(image=img, mask=mask, bboxes=boxes, labels=labels)

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
show(ax[0], img, boxes, mask, "Original")
show(ax[1], t["image"], t["bboxes"], t["mask"], "Geometric aug (flip + rotate 25)")
fig.suptitle("Geometric transform -> labels (boxes + mask) move WITH the image")
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig("outputs/04_geometric.png", dpi=130, bbox_inches="tight")
plt.show()
print("저장: outputs/04_geometric.png  (기하 변환 -> 박스·마스크도 같이 움직인다)")

In [ ]:
# 셀 5 · 픽셀 증강 (라벨 그대로) — 이미지에만 적용, 박스·마스크 좌표는 유지
import os, json, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import albumentations as A
from PIL import Image

demo = os.path.join("data", "_demo_inputs")
img = np.array(Image.open(os.path.join(demo, "aug_image.jpg")).convert("RGB"))
mask = (np.array(Image.open(os.path.join(demo, "aug_mask.png")).convert("L")) > 127).astype(np.uint8)
d = json.load(open(os.path.join(demo, "aug_boxes.json")))
boxes, labels = d["boxes"], d["labels"]


def show(ax, image, bxs, m, title):
    disp = image.copy()
    if m.shape == disp.shape[:2]:                       # 크기 일치할 때만 마스크 오버레이(방어)
        sel = m > 0
        disp[sel] = (0.5 * disp[sel] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
    ax.imshow(disp)
    for bx in bxs:
        x1, y1, x2, y2 = [int(v) for v in bx]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=1.5))
    ax.set_title(title)
    ax.axis("off")


# 픽셀 변환은 좌표를 바꾸지 않는다(밝기/대비/색조). 재현성 위해 시드 고정.
pixel = [A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
         A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=20, val_shift_limit=10, p=1.0)]
try:
    transform = A.Compose(pixel, seed=0)               # albumentations>=2.0
except TypeError:
    random.seed(0); np.random.seed(0)
    transform = A.Compose(pixel)
aug = transform(image=img)["image"]

# 같은 박스/마스크를 원본과 증강 양쪽에 '그대로' 그려서 라벨이 안 변함을 보여준다
fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
show(ax[0], img, boxes, mask, "Original")
show(ax[1], aug, boxes, mask, "Pixel aug (brightness/hue) - SAME labels")
fig.suptitle("Pixel transform -> image changes, labels (boxes + mask) stay PUT")
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig("outputs/04_pixel.png", dpi=130, bbox_inches="tight")
plt.show()
print("저장: outputs/04_pixel.png  (픽셀 변환 -> 이미지만 바뀌고 라벨 좌표는 그대로)")

In [ ]:
# 셀 6 · 나쁜 증강 경고 — 도메인 부적합 변환(극단적 색조/채도 -> 염색 색 정보 파괴)
import os
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from PIL import Image

demo = os.path.join("data", "_demo_inputs")
img = np.array(Image.open(os.path.join(demo, "aug_image.jpg")).convert("RGB"))

# 결정적 극단 변환: 색조를 크게 돌리고 채도를 크게 낮춤 -> 혈액 도말의 '염색 색'이 파괴됨
bad = A.Compose([A.HueSaturationValue(hue_shift_limit=(90, 90), sat_shift_limit=(-70, -70),
                                      val_shift_limit=0, p=1.0)])
out = bad(image=img)["image"]

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].imshow(img); ax[0].set_title("Original (stain color = info)"); ax[0].axis("off")
ax[1].imshow(out); ax[1].set_title("BAD aug: extreme hue/sat shift"); ax[1].axis("off")
fig.suptitle("Bad augmentation: domain-inappropriate color shift destroys stain info -> HURTS the model")
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig("outputs/04_bad_aug.png", dpi=130, bbox_inches="tight")
plt.show()
print("저장: outputs/04_bad_aug.png")
print("경고: 색이 곧 정보인 생물 이미지(염색 도말)에서 극단적 색조/채도 증강은 의미를 파괴 -> 오히려 해롭다.")

In [ ]:
# 셀 7 · 애니메이션 (HTML용) — 점진적 회전 + 마스크가 따라 움직이는 프레임 -> GIF
import os
import numpy as np
import albumentations as A
from PIL import Image

demo = os.path.join("data", "_demo_inputs")
img = np.array(Image.open(os.path.join(demo, "aug_image.jpg")).convert("RGB"))
mask = (np.array(Image.open(os.path.join(demo, "aug_mask.png")).convert("L")) > 127).astype(np.uint8)

frames = []
for ang in range(0, 31, 5):                 # 0,5,...,30도 (마스크도 같은 각도로 함께 회전)
    t = A.Compose([A.Affine(rotate=(ang, ang), p=1.0)])(image=img, mask=mask)
    fr = t["image"].copy()
    sel = t["mask"] > 0
    fr[sel] = (0.5 * fr[sel] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)   # 마스크 빨강 오버레이
    frames.append(Image.fromarray(fr))

seq = frames + frames[::-1]                  # 왕복으로 부드럽게
os.makedirs("outputs", exist_ok=True)
gif = os.path.join("outputs", "04_aug_animation.gif")
seq[0].save(gif, save_all=True, append_images=seq[1:], duration=120, loop=0)
frames[0].save(os.path.join("outputs", "04_aug_frame_first.png"))
frames[-1].save(os.path.join("outputs", "04_aug_frame_last.png"))
print("저장:", gif, "(프레임", len(seq), "장) + 미리보기 PNG 2장")
print("=> 기하 변환에서 마스크(라벨)가 이미지를 따라 함께 회전하는 것을 애니메이션으로 보여준다. (5단계 HTML 재사용)")

In [ ]:
# 셀 8 · 종합 / 검증
import os

figs = ["outputs/04_geometric.png", "outputs/04_pixel.png", "outputs/04_bad_aug.png"]
gif = "outputs/04_aug_animation.gif"
print("=" * 56)
print(" 4단계 · 증강 검증/요약")
print("=" * 56)
for f in figs:
    print(" -", os.path.basename(f).ljust(20), ":", "있음" if os.path.exists(f) else "없음")
print(" - 애니메이션 GIF       :", "있음" if os.path.exists(gif) else "없음", "(", gif, ")")
if not all(os.path.exists(f) for f in figs + [gif]):
    print(" [주의] 일부 결과가 없습니다 — 셀 3~7 을 순서대로 다시 ▶ 실행하세요.")
print("=" * 56)
print("배운 것:")
print(" 1) 기하 변환(플립/회전/어파인) -> 라벨(박스·마스크)도 함께 이동")
print(" 2) 픽셀 변환(밝기/대비/색조)   -> 이미지만 변하고 라벨 좌표는 그대로")
print(" 3) 나쁜 증강(도메인 부적합)    -> 색이 정보인 생물 이미지에선 오히려 해로움")
print("참고: 합성 증강(copy-paste 로 새 예시 만들기)도 있으나 도메인 적합성 검증이 필요하다(개념).")